# ViMD — tải và nghe 5 utterance lỗi nặng

Notebook này lấy đúng 5 file audio trong phần phân tích lỗi của PhoWhisper-large trên
[ViMD](https://huggingface.co/datasets/nguyendv02/ViMD_Dataset), rồi **phát ngay trong trình duyệt**
kèm reference và predicted để đối chiếu.

| file | tỉnh | thời lượng | WER | kiểu lỗi |
|---|---|---|---|---|
| `73_0332.wav` | Quảng Bình | 17.9 s | 452 % | repetition loop |
| `81_0303.wav` | Gia Lai | 11.8 s | 398 % | repetition loop |
| `77_0282.wav` | Bình Định | 21.8 s | 95 % | chèn đoạn mở đầu |
| `76_0327.wav` | Quảng Ngãi | 11.5 s | 73 % | mất nửa đầu |
| `38_0284.wav` | Hà Tĩnh | 13.7 s | 49 % | nghe nhầm phương ngữ |

**Cách dùng:** `Runtime → Run all` (Ctrl+F9). Mất khoảng 1–2 phút, không cần GPU, không cần đăng nhập.

**Không tải 60 GB.** ViMD publish dưới dạng parquet ~60 GB. Parquet là định dạng cột và chia thành
row group, nên notebook mở shard *qua mạng*, chỉ đọc cột `filename` để định vị utterance, rồi kéo
đúng một row group chứa nó — vài chục MB thay vì vài chục GB. Bytes WAV được ghi ra nguyên vẹn nên
audio giống hệt bản gốc trên Hub.

> ViMD có giấy phép **CC-BY-NC-ND-4.0** — chỉ dùng cho nghiên cứu phi thương mại.

## 1. Cài thư viện

In [ ]:
# Colab thường có sẵn, chạy cho chắc. %pip (không phải !pip) cài đúng vào kernel đang chạy.
%pip install -q huggingface_hub pyarrow soundfile

## 2. Cấu hình và bảng phân tích lỗi

In [ ]:
from pathlib import Path

DATASET_ID = "nguyendv02/ViMD_Dataset"
REVISION = ""      # để trống = HEAD hiện tại trên Hub; điền sha để cố định
HF_TOKEN = ""      # ViMD là dataset public, để trống là được
SPLITS = ("test", "valid", "train")   # quét test trước: cả 5 mẫu đều nằm ở test
OUT_DIR = Path("vimd_samples")
ALSO_16K = False   # True: ghi thêm bản mono 16 kHz - đúng tín hiệu model nghe

# Reference / predicted chép từ bảng phân tích lỗi. `note` mô tả phần đuôi bị lặp
# đến hết giới hạn 448 token, không tiện chép nguyên.
SAMPLES = {
    "73_0332.wav": {
        "error": "Repetition loop",
        "province": "Quảng Bình · Central",
        "wer": "452 %",
        "reference": (
            "Cảm ơn Nhà nước. Cảm ơn anh em chị em. Các chú các con sang với mệ. Mệ cảm ơn. "
            "Nhưng mà có cái tiền bạc ăn uống lương tháng cũng nhờ Nhà nước. Vậy chứ mệ mới có ăn có tiêu."
        ),
        "predicted": (
            "Cảm ơn nhà nước. Cảm ơn anh em, chị em, các chú, các con đã tham gia việc với mình. "
            "Cảm ơn. Hy vọng của cái liên mạc an hưởng nước bạn cũng như nhà nước. Vậy chứ mấy người, "
            "cỏ ăn, cỏ thiếu, ủng hộ, ủng hộ ủng hộ ủng hộ …"
        ),
        "note": "“ủng hộ” ×26, rồi “hội” ×110, chạy tới trần 448 token",
    },
    "81_0303.wav": {
        "error": "Repetition loop",
        "province": "Gia Lai · Central",
        "wer": "398 %",
        "reference": (
            "Cái giống là BĐR ba con chín nó rất hiệu quả, còn giống bốn tám hồi giờ nó tương tự "
            "giống như ông Giang nói. Nhưng mà làm gì làm mà biết chuyển đổi cây trồng thì bà con "
            "rất là vui mừng"
        ),
        "predicted": (
            "Cây giông là BDR, Bệnh viện kiên đã từng mang tất nó rất hiệu quả, còn bù la tang bây giờ "
            "nó thường xuyên giống như ông Giang nói, nhưng mà làm gì làm mà không ít, thế được cây trồng "
            "bà con rất là mừng. Cây trồng bà con rất là vui, chúng tôi rất là vui khi được trồng cây này …"
        ),
        "note": "“được trồng cây này” ×34, chạy tới trần token",
    },
    "77_0282.wav": {
        "error": "Chèn đoạn mở đầu không có trong audio",
        "province": "Bình Định · Central",
        "wer": "95 %",
        "reference": (
            "Bớt chi phí cho phân bón, bớt cho lao động nhưng mà năng suất vẫn cao. Hiệu quả là hiệu quả "
            "cái đó. Ngày xưa là bốn lần hoặc ba lần thì cái công nó sẽ nhiều hơn. Tăng cái chi phí lên. "
            "Nhưng bây giờ đây giảm chi phí mà năng suất cao thì có lợi cho dân"
        ),
        "predicted": (
            "Đến giờ này thì phải nói là bà con thoải mái. Năng suất ánh nồng ngon nhất là phải tám mươi "
            "sáu trạng trên một trạm. Mẹ đứt cần phải tám mươi trăm bốn chục cân trên một trạm. Ờ bớt chi phí, "
            "phân bón, bớt lao động. Nhưng mà năng suất vẫn cao…"
        ),
        "note": "phần in đậm ở đầu không hề có trong audio; đoạn khớp reference chỉ bắt đầu từ “Ờ bớt chi phí”",
    },
    "76_0327.wav": {
        "error": "Bỏ mất nửa đầu",
        "province": "Quảng Ngãi · Central",
        "wer": "73 %",
        "reference": (
            "Giờ lo xót bên ảnh thôi, lo cho ảnh từ đầu đến cuối thôi chứ, chứ đâu bên kia có tự được "
            "thứ gì hết. Ai ăn uống cũng đút, chứ cũng không tự xúc ăn được."
        ),
        "predicted": "Ai ăn uống cũng đốt, chứ cũng không tự sốc ăn được.",
        "note": "chỉ ra đúng câu cuối; toàn bộ nửa đầu bị bỏ",
    },
    "38_0284.wav": {
        "error": "Nghe nhầm phương ngữ",
        "province": "Hà Tĩnh · Central",
        "wer": "49 %",
        "reference": (
            "Cái cấp phép bể bơi đây á, mà của mình thì hiện tại nó đang ở một cửa. Thành ra là khi mình "
            "đã trình hồ sơ một cửa rồi nhưng mà chưa có hồ sơ giấy phép về."
        ),
        "predicted": (
            "Cặp chép để bơ đấy ạ. Mô của chị là hiện tại là đang ở một cửa. Thành ra là khi khi khi khi khi "
            "mình đã trình hồ sơ một cửa rồi nhưng mà chưa bỏ hồ sơ giấy khép phế á."
        ),
        "note": "“cấp phép bể bơi” → “cặp chép để bơ”; kèm một đoạn lặp ngắn “khi khi khi”",
    },
}

FILENAMES = list(SAMPLES)
print(f"{len(FILENAMES)} file cần lấy: {', '.join(FILENAMES)}")

## 3. Hàm phụ: giải mã WAV, đọc cột audio của parquet

In [ ]:
import io
import math
import os
import re
import time
import wave
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

# Tên shard ViMD trên Hub, ví dụ data/test-00000-of-00003.parquet
_SHARD_NAME_RE = re.compile(r"^data/(?P<split>train|valid|test)-(?P<index>\d+)-of-\d+\.parquet$")

METADATA_COLUMNS = ("region", "province_code", "province_name", "filename", "text", "speakerID", "gender")
TARGET_RATE = 16000


def _key(filename: str) -> str:
    """Khoá so khớp: bỏ thư mục, bỏ đuôi .wav, hạ chữ thường."""
    stem = Path(str(filename).strip().replace("\\", "/")).name
    if stem.lower().endswith(".wav"):
        stem = stem[:-4]
    return stem.lower()


def _retry(operation, description: str, attempts: int = 4):
    """Thử lại với backoff tăng dần - mạng Colab thỉnh thoảng chập chờn."""
    last = None
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except (KeyboardInterrupt, SystemExit):
            raise
        except BaseException as exc:
            last = exc
            wait = min(30.0, 3.0 * (2 ** (attempt - 1)))
            print(f"  ! {description} lỗi lần {attempt}/{attempts}: {exc}", flush=True)
            if attempt < attempts:
                time.sleep(wait)
    raise RuntimeError(f"{description} thất bại sau {attempts} lần") from last


def _audio_bytes_from_column(column: Any, row: int) -> bytes:
    """ViMD lưu audio là HuggingFace Audio feature, tức struct {bytes, path}."""
    import pyarrow as pa

    # Đọc một row group trả về Table nên cột là ChunkedArray - không có .field().
    if isinstance(column, pa.ChunkedArray):
        combined = column.combine_chunks()
        if isinstance(combined, pa.ChunkedArray):
            if combined.num_chunks == 0:
                raise RuntimeError("cột audio rỗng")
            combined = combined.chunk(0)
        column = combined
    if pa.types.is_struct(column.type):
        value = column.field("bytes")[row].as_py()
        if value is None:
            raise RuntimeError(f"row audio không có bytes nội tuyến (path={column.field('path')[row].as_py()!r})")
        return value
    value = column[row].as_py()
    if value is None:
        raise RuntimeError("row audio rỗng")
    return value


def _wav_shape(raw: bytes) -> Tuple[Optional[float], Optional[int], Optional[int]]:
    """(giây, sample rate, số kênh) đọc từ header - không giải mã mẫu nào."""
    try:
        with wave.open(io.BytesIO(raw), "rb") as handle:
            rate, frames, channels = handle.getframerate(), handle.getnframes(), handle.getnchannels()
        return (float(frames) / rate if rate else None, int(rate), int(channels))
    except Exception:
        try:
            import soundfile as sf

            info = sf.info(io.BytesIO(raw))
            return (float(info.duration), int(info.samplerate), int(info.channels))
        except Exception:
            return (None, None, None)


def _decode_wav_bytes(raw: bytes) -> Tuple[np.ndarray, int]:
    """Giải mã WAV thành float32 trong [-1, 1], kèm sample rate gốc."""
    try:
        import soundfile as sf

        data, rate = sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)
        return np.asarray(data, dtype=np.float32), int(rate)
    except Exception:
        with wave.open(io.BytesIO(raw), "rb") as handle:
            channels, width, rate = handle.getnchannels(), handle.getsampwidth(), handle.getframerate()
            frames = handle.readframes(handle.getnframes())
        if width != 2:
            raise RuntimeError(f"độ rộng mẫu WAV không hỗ trợ: {width} byte")
        samples = np.frombuffer(frames, dtype="<i2").astype(np.float32) / 32768.0
        return (samples.reshape(-1, channels) if channels > 1 else samples), rate


def _to_target_mono_int16(samples: np.ndarray, rate: int, target_rate: int = TARGET_RATE) -> np.ndarray:
    """Trộn mono + resample - đúng phép biến đổi pipeline áp lên dữ liệu huấn luyện."""
    from scipy.signal import resample_poly

    if samples.ndim == 2:
        samples = samples.mean(axis=1)
    samples = np.asarray(samples, dtype=np.float32)
    if int(rate) != int(target_rate):
        divisor = math.gcd(int(rate), int(target_rate))
        samples = resample_poly(samples, target_rate // divisor, int(rate) // divisor).astype(np.float32)
    return np.rint(np.clip(samples, -1.0, 1.0) * 32767.0).astype("<i2")


def _write_wav_int16(path: Path, pcm: np.ndarray, rate: int = TARGET_RATE) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(path), "wb") as handle:
        handle.setnchannels(1)
        handle.setsampwidth(2)
        handle.setframerate(int(rate))
        handle.writeframes(np.asarray(pcm, dtype="<i2").tobytes())

print("helpers sẵn sàng")

## 4. Tìm và lấy utterance từ parquet trên Hub

`_ShardReader` mở shard qua `HfFileSystem` (đọc theo range request, không tải cả file). Nếu range
request bị chặn thì tự chuyển sang tải nguyên shard, quét xong xoá ngay — chậm hơn nhiều nhưng vẫn chạy.

In [ ]:
class _ShardReader:
    """Mở shard parquet trên Hub. Ưu tiên streaming; tải nguyên shard là phương án cuối."""

    def __init__(self, dataset_id: str, revision: str, token: Optional[str] = None) -> None:
        self.dataset_id, self.revision, self.token = dataset_id, revision, token or None
        self._fs = None
        self._streaming = True
        self._local: Dict[str, Path] = {}

    def open(self, filename: str):
        """(ParquetFile, hàm_đóng). File tải về được giữ tới khi gọi release()."""
        import pyarrow.parquet as pq

        if self._streaming:
            try:
                if self._fs is None:
                    from huggingface_hub import HfFileSystem

                    self._fs = HfFileSystem(token=self.token)
                path = f"datasets/{self.dataset_id}@{self.revision}/{filename}"
                # Chỉ thử 2 lần: nếu range request không dùng được thì fallback bên dưới
                # mới là chỗ đáng retry, không phải ở đây.
                handle = _retry(lambda: self._fs.open(path, "rb"), f"mở {filename} qua mạng", attempts=2)
                return pq.ParquetFile(handle), handle.close
            except (KeyboardInterrupt, SystemExit):
                raise
            except BaseException as exc:
                print(f"  ! không đọc trực tiếp được ({exc}); chuyển sang tải nguyên shard", flush=True)
                self._streaming = False

        from huggingface_hub import hf_hub_download

        local = self._local.get(filename)
        if local is None or not local.exists():
            local = Path(_retry(
                lambda: hf_hub_download(
                    repo_id=self.dataset_id, filename=filename, repo_type="dataset",
                    revision=self.revision, local_dir="_vimd_scratch",
                ),
                f"tải {filename}",
            ))
            self._local[filename] = local
        parquet_file = pq.ParquetFile(str(local))
        return parquet_file, parquet_file.close

    def release(self, filename: Optional[str] = None) -> None:
        """Xoá shard đã tải. Ở chế độ streaming thì không có gì để xoá."""
        for name in ([filename] if filename else list(self._local)):
            local = self._local.pop(name, None)
            if local is not None:
                try:
                    local.unlink()
                except OSError:
                    pass


def _shard_list(dataset_id: str, revision: str, token: Optional[str], splits: Sequence[str]):
    """[(split, tên shard)] theo đúng thứ tự split muốn quét."""
    from huggingface_hub import HfApi

    files = _retry(
        lambda: HfApi(token=token or None).list_repo_files(dataset_id, repo_type="dataset", revision=revision),
        f"liệt kê file của {dataset_id}",
    )
    grouped: Dict[str, List[Tuple[int, str]]] = {}
    for name in files:
        match = _SHARD_NAME_RE.match(name)
        if match:
            grouped.setdefault(match.group("split"), []).append((int(match.group("index")), name))
    ordered = [(s, n) for s in splits for _, n in sorted(grouped.get(s, []))]
    if not ordered:
        raise RuntimeError(f"{dataset_id}@{revision} không có shard parquet nào khớp mẫu tên mong đợi")
    return ordered


def fetch_samples(filenames, dataset_id=None, revision=None, token=None, splits=None,
                  out_dir=None, also_16k=None) -> Dict[str, Dict[str, Any]]:
    """Tải các utterance được nêu tên và ghi ra WAV. Trả về {khoá: thông tin}."""
    dataset_id = dataset_id or DATASET_ID
    token = token if token is not None else HF_TOKEN
    splits = tuple(splits or SPLITS)
    out_dir = Path(out_dir or OUT_DIR)
    also_16k = ALSO_16K if also_16k is None else also_16k

    if not revision:
        from huggingface_hub import HfApi

        revision = _retry(lambda: HfApi(token=token or None).dataset_info(dataset_id).sha,
                          f"tra cứu {dataset_id}")
    print(f"dataset {dataset_id} @ {revision}", flush=True)

    wanted = {_key(name) for name in filenames}
    out_dir.mkdir(parents=True, exist_ok=True)
    reader = _ShardReader(dataset_id, revision, token)
    results: Dict[str, Dict[str, Any]] = {}

    try:
        for split, shard in _shard_list(dataset_id, revision, token, splits):
            if not wanted:
                break
            try:
                parquet_file, close = reader.open(shard)
                try:
                    columns = parquet_file.schema_arrow.names
                    if "audio" not in columns or "filename" not in columns:
                        raise RuntimeError(f"{shard} thiếu cột audio/filename; schema là {sorted(columns)}")
                    present = [c for c in METADATA_COLUMNS if c in columns]

                    # Chỉ đọc cột `filename` của từng row group - vài chục KB, đủ để định vị.
                    hits: Dict[int, List[Tuple[str, int, str]]] = {}
                    for group in range(parquet_file.num_row_groups):
                        names = parquet_file.read_row_group(group, columns=["filename"]).column("filename").to_pylist()
                        for row, name in enumerate(names):
                            key = _key(name or "")
                            if key in wanted:
                                hits.setdefault(group, []).append((key, row, str(name)))
                    if not hits:
                        print(f"  {shard}: không có file cần tìm", flush=True)
                        continue

                    found_here = [k for items in hits.values() for k, _, _ in items]
                    print(f"  {shard}: thấy {', '.join(sorted(found_here))}", flush=True)

                    # Chỉ kéo đúng những row group chứa hit - đây là chỗ tiết kiệm băng thông.
                    for group, items in sorted(hits.items()):
                        table = parquet_file.read_row_group(group, columns=["audio", *present])
                        audio_column = table.column("audio")
                        metadata = {name: table.column(name).to_pylist() for name in present}
                        for key, row, name in items:
                            raw = _audio_bytes_from_column(audio_column, row)
                            duration, rate, channels = _wav_shape(raw)
                            path = out_dir / f"{key}.wav"
                            path.write_bytes(raw)  # nguyên bytes trong parquet, không mã hoá lại
                            entry = {
                                "filename": name, "split": split, "shard": shard, "path": str(path),
                                "duration": duration, "sampling_rate": rate, "channels": channels,
                                "reference": metadata.get("text", [None] * (row + 1))[row] if "text" in metadata else None,
                            }
                            entry.update({n: metadata[n][row] for n in present if n not in ("filename", "text")})
                            if also_16k:
                                samples, source_rate = _decode_wav_bytes(raw)
                                path16 = out_dir / f"{key}.16k.wav"
                                _write_wav_int16(path16, _to_target_mono_int16(samples, source_rate))
                                entry["path_16k"] = str(path16)
                            results[key] = entry
                            wanted.discard(key)
                finally:
                    close()
            finally:
                reader.release(shard)
    finally:
        reader.release()

    if wanted:
        print(f"\n! không tìm thấy trong split {'/'.join(splits)}: {', '.join(sorted(wanted))}", flush=True)
    return results

print("fetch_samples() sẵn sàng")

## 5. Tải về

In [ ]:
results = fetch_samples(FILENAMES)

print()
for key in (_key(n) for n in FILENAMES):
    entry = results.get(key)
    if entry is None:
        continue
    duration = entry.get("duration")
    print(
        f"{entry['filename']:<14} {str(entry.get('province_name') or ''):<12} "
        f"{str(entry.get('region') or ''):<9} "
        f"{duration:>5.1f}s  {entry.get('sampling_rate')} Hz  "
        f"{entry.get('channels')} kênh  ->  {entry['path']}"
    )
print(f"\n{len(results)}/{len(FILENAMES)} file đã tải về {OUT_DIR}/")

## 6. Nghe

Mỗi mẫu: player, reference của dataset, predicted của model, và ghi chú về kiểu lỗi.

In [ ]:
import html as _html

from IPython.display import Audio, HTML, display

BOX = (
    "border-left:4px solid {color};background:rgba(127,127,127,0.08);"
    "padding:10px 14px;margin:18px 0 6px;border-radius:4px;line-height:1.55"
)
COLORS = {"Repetition loop": "#dc2626"}


def _row(label, text, color="inherit"):
    return (
        f"<div style='margin-top:6px'><span style='opacity:.6;font-size:.85em;"
        f"letter-spacing:.03em'>{label}</span><br>"
        f"<span style='color:{color}'>{_html.escape(str(text))}</span></div>"
    )


for name in FILENAMES:
    key = _key(name)
    entry = results.get(key)
    meta = SAMPLES[name]
    if entry is None:
        display(HTML(f"<div style='{BOX.format(color='#9ca3af')}'><b>{name}</b> — chưa tải được</div>"))
        continue

    duration = entry.get("duration")
    header = (
        f"<b style='font-size:1.05em'>{name}</b> &nbsp;·&nbsp; {_html.escape(meta['province'])}"
        f" &nbsp;·&nbsp; {duration:.1f} s &nbsp;·&nbsp; WER <b>{meta['wer']}</b>"
        f" &nbsp;·&nbsp; {_html.escape(meta['error'])}"
    )
    body = header + _row("REFERENCE", entry.get("reference") or meta["reference"])
    body += _row("PREDICTED", meta["predicted"], COLORS.get(meta["error"], "inherit"))
    if meta.get("note"):
        body += (
            f"<div style='margin-top:6px;opacity:.7;font-size:.9em'>↳ {_html.escape(meta['note'])}</div>"
        )
    display(HTML(f"<div style='{BOX.format(color=COLORS.get(meta['error'], '#2563eb'))}'>{body}</div>"))
    display(Audio(entry["path"]))

## 7. Lấy file về máy

In [ ]:
import shutil

archive = shutil.make_archive("vimd_samples", "zip", OUT_DIR)
print(f"{archive}  ({Path(archive).stat().st_size / 1e6:.1f} MB)")

try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print("(không chạy trong Colab - file zip nằm sẵn ở đường dẫn trên)")

### Tuỳ chọn: chép sang Google Drive

In [ ]:
# Bỏ dấu # ở các dòng dưới nếu muốn giữ file lại trên Drive thay vì tải về máy.
# from google.colab import drive
# drive.mount("/content/drive")
# destination = Path("/content/drive/MyDrive/vimd_samples")
# destination.mkdir(parents=True, exist_ok=True)
# for wav in sorted(OUT_DIR.glob("*.wav")):
#     shutil.copy2(wav, destination / wav.name)
#     print("->", destination / wav.name)

---

### Ghi chú

- **Muốn nghe utterance khác?** Sửa `FILENAMES` ở cell 5, ví dụ
  `results = fetch_samples(["12_0001", "45_0123.wav"])`. Tên chính là cột `filename` trong
  `outputs/test_predictions.csv`, có hay không có đuôi `.wav` đều được.
- **Audio là bản gốc**, 44.1 kHz stereo, bytes ghi nguyên vẹn từ parquet. Đặt `ALSO_16K = True`
  để có thêm bản mono 16 kHz — đúng tín hiệu model nhận vào, hữu ích khi nghi ngờ lỗi đến từ
  khâu resample chứ không phải từ model.
- **Chạy dưới local thay vì Colab?** Repo có `download_samples.py` làm đúng việc này ở dạng CLI,
  và còn đọc được cache 16 kHz do `main.py` sinh ra mà không cần mạng.
- Notebook cố ý tự chứa: không import gì từ repo, mở link Colab là chạy được ngay.